# Etna Dataset Construction

This notebook builds the canonical hourly Etna dataset used by the Cause–Trigger analysis. The saved CSV remains in physical or proxy units; transformations and standardization are applied later to the selected reference and case intervals.

In [1]:
import sys
from pathlib import Path
from obspy.clients.fdsn import Client
from obspy.clients.fdsn import Client
from obspy.io.mseed import InternalMSEEDWarning
import warnings
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
SRC_DIR = PROJECT_ROOT / "src/etna"
sys.path.append(str(SRC_DIR))

from etna_config import (
    EVENT_TIME,
    ETNA_GAS_METEO_COLS,
    ETNA_WEATHER_COLS,
    ETNA_WAVEFORM_CONFIG,
    etna_observable_metadata,
)
from etna_dataset import (
    create_etna_dataset,
    load_etna_event_catalog_xls,
    load_etnagas_csv,
    load_openmeteo_etna_weather,
)
from etna_plotting_utils import (
    dataset_health_report,
    distribution_summary,
    plot_etna_all_variables_map,
    plot_etna_thesis_figures,
    run_teleseismic_checks,
    plot_etna_loglog_distributions,
)
from etna_waveform import build_station_waveform_dataset

warnings.filterwarnings(
    "ignore",
    category=InternalMSEEDWarning,
    message=r".*fractional second.*10000.*",
)

## Hourly teleseismic waveform proxy

ESLN HHZ data are processed in padded daily chunks. The proxy is the hourly maximum of non-overlapping 120-second RMS values in the 0.03–0.30 Hz band.

In [2]:
waveform_df, waveform_failures = build_station_waveform_dataset(
    client=Client("INGV"),
    station=ETNA_WAVEFORM_CONFIG["station"],
    config=ETNA_WAVEFORM_CONFIG,
    cache_path="../data/etna/etna_esln_hhz_waveform.pkl",
    redownload=True, # True to re-download of waveform data, False to use cached data
)

print(f"Waveform rows: {len(waveform_df)}")
print(f"Failed daily chunks: {len(waveform_failures)}")
if waveform_failures:
    display(pd.DataFrame(waveform_failures, columns=["date", "error"]))

ESLN OK   2008-04-12
ESLN OK   2008-04-13
ESLN OK   2008-04-14
ESLN OK   2008-04-15
ESLN OK   2008-04-16
ESLN OK   2008-04-17 | missing hourly values=1
ESLN OK   2008-04-18 | missing hourly values=2
ESLN OK   2008-04-19 | missing hourly values=1
ESLN OK   2008-04-20
ESLN OK   2008-04-21 | missing hourly values=1


c:\Users\cescedes\anaconda3\envs\M\Lib\site-packages\obspy\io\mseed\util.py:683: UserWarning: Record contains a fractional seconds (.0001 secs) of 10000 - the maximum strictly allowed value is 9999. It will be interpreted as one or more additional seconds.
  warnings.warn(


ESLN OK   2008-04-22 | missing hourly values=2
ESLN OK   2008-04-23 | missing hourly values=1
ESLN OK   2008-04-24
ESLN OK   2008-04-25
ESLN OK   2008-04-26
ESLN OK   2008-04-27


KeyboardInterrupt: 

## External variables

The retained external variables are soil CO₂ concentration and atmospheric pressure drop from ETNAGAS, plus hourly Open-Meteo precipitation. Isolated one-hour ETNAGAS gaps are linearly interpolated and reported explicitly.

Liuzzo, M., Giuffrida, G. B., & Gurrieri, S. (2025). *Etna CO2 Soil Flux during 2002–2010 (ECSF2002_2010)*. INGV. https://doi.org/10.13127/etna/ecsf2002_2010

Zippenfenig, P. (2023). *Open-Meteo.com Weather API*. Zenodo. https://doi.org/10.5281/ZENODO.7970649

In [ ]:
etnagas_df, etnagas_interpolation_report = load_etnagas_csv(
    "../data/etna/3c.csv",
    value_cols=ETNA_GAS_METEO_COLS,
    start_time=ETNA_WAVEFORM_CONFIG["start"].isoformat(),
    end_time=ETNA_WAVEFORM_CONFIG["end"].isoformat(),
)

display(etnagas_interpolation_report)

In [ ]:
weather_cache = "../data/etna/etna_openmeteo_hourly.csv"

if weather_cache.exists():
    weather_df = pd.read_csv(
        weather_cache,
        parse_dates=["timestamp"],
    )
    weather_df["timestamp"] = pd.to_datetime(
        weather_df["timestamp"],
        utc=True,
    )
else:
    weather_df = load_openmeteo_etna_weather(
        start_date="2008-04-12",
        end_date="2008-05-15",
    )
    weather_df.to_csv(weather_cache, index=False)

print(f"Weather rows: {len(weather_df)}")

## Mt. Etna Seismic Catalogue 2000–2010

The catalogue supplies two hourly variables:

- `local_event_rate_state`: a past-only 48-hour local-seismicity state ending six hours before the current hour;
- `local_event_rate_response`: a positive six-hour event-count response relative to an earlier past baseline, used as the effect variable.

Alparone, S. C., Maiolino, V., Mostaccio, A., Scaltrito, A., Ursino, A., Barberi, G., et al. (2015). *Mt. Etna Seismic Catalog 2000–2010* [Data set]. INGV–Osservatorio Etneo. https://doi.org/10.13127/etnasc/2000_2010

In [ ]:
catalogue = load_etna_event_catalog_xls(
    "../data/etna/Etna catalogue_2000-2010.xls",
    quality_filter=False,
)

print(f"Catalogue rows: {len(catalogue)}")
print(
    "Catalogue range:",
    catalogue["timestamp"].min(),
    "to",
    catalogue["timestamp"].max(),
)

## Save the hourly dataset

All sources are joined by exact UTC hourly timestamps. The first rows lacking the required past-only catalogue windows are removed; no other missing rows are silently filled during the final merge.

In [ ]:
etna_dataset = create_etna_dataset(
    wave_df=waveform_df,
    start_time=ETNA_WAVEFORM_CONFIG["start"].isoformat(),
    end_time=ETNA_WAVEFORM_CONFIG["end"].isoformat(),
    catalog_df=catalogue,
    etnagas_df=etnagas_df,
    etnagas_cols=ETNA_GAS_METEO_COLS,
    weather_df=weather_df,
    weather_cols=ETNA_WEATHER_COLS,
    output_dir="../data/etna",
)

## Construction audit

This compact audit reports the final hourly-grid integrity, waveform failures, the leading rows removed for past-only proxy warm-up, and ETNAGAS interpolation count.

In [ ]:
expected_hours = int(
    (ETNA_WAVEFORM_CONFIG["end"] - ETNA_WAVEFORM_CONFIG["start"]) / 3600
)

audit = dataset_health_report(
    etna_dataset,
    "Etna canonical hourly dataset",
)
audit["waveform_failed_chunks"] = len(waveform_failures)
audit["proxy_warmup_rows_removed"] = expected_hours - len(etna_dataset)
audit["etnagas_interpolated_hours"] = len(
    etnagas_interpolation_report
)

display(audit)
display(distribution_summary(etna_dataset))

### Distribution diagnostics

Empirical density plots and summary statistics are used to inspect skewness, outliers, and scaling behavior. 


In [ ]:
plot_variable_pdfs(etna_dataset, filename="etna_pdf")

In [ ]:
summary_esln = distribution_summary(etna_dataset)
display(summary_esln)

### Log-log distribution diagnostics

These diagnostics use the canonical unscaled variables. True logarithmic axes require positive values, so zeros and negative observations are excluded variable by variable and reported explicitly.

In [ ]:
fig, axes, etna_loglog_report = plot_etna_loglog_distributions(
    dataframe=etna_dataset,
    save_dir="figures",
)

display(etna_loglog_report)

### Final overview figures

In [ ]:
plot_etna_thesis_figures(
    csv_path="../data/etna/etna_dataset.csv",
    event_time=EVENT_TIME,
    save_dir="figures",
    include_titles=False,
)

### Exact teleseismic-feature diagnostic

The following figures reuse the same response correction, frequency band, 120-second RMS windows, and hourly-maximum aggregation used in the canonical dataset.

In [ ]:
waveform_client = Client(
    base_url="https://webservices.ingv.it",
    service_mappings={
        "dataselect": "https://webservices.ingv.it/fdsnws/dataselect/1/",
        "station": "https://webservices.ingv.it/fdsnws/station/1/",
        "event": "https://webservices.ingv.it/fdsnws/event/1/",
    },
    _discover_services=False,
)

teleseismic_check = run_teleseismic_checks(
    client=waveform_client,
    station=ETNA_WAVEFORM_CONFIG["station"],
    config=ETNA_WAVEFORM_CONFIG,
    event_time=EVENT_TIME,
    save_dir="figures",
)

### Geographic Visualization

In [ ]:
figure, axis, source_table = plot_etna_all_variables_map(
    metadata=etna_observable_metadata(),
    satellite=True,
    save_dir="figures",
    filename="etna_map",
)

display(source_table)